Books to Scrape
      ↓
Scrape first 5 catalogue pages
      ↓
100 books collected
      ↓
Select 60 books from at least 3 categories
      ↓
Clean price, rating, availability
      ↓
Calculate price_inr = price_gbp × 105.50
      ↓
Create SQLite database
      ↓
Run 5 SQL queries
      ↓
Read results using pd.read_sql()
      ↓
Reproduce JOIN using pd.merge()
      ↓
Compare both results

In [ ]:
pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
import os
from urllib.parse import urljoin
from IPython.display import display

BASE_URL = "https://books.toscrape.com/"
ALL_PRODUCTS_URL = "https://books.toscrape.com/catalogue/page-{}.html"

EXCHANGE_RATE = 105.50

REQUIRED_BOOKS = 60
PAGES_TO_SCRAPE = 5

DB_NAME = "books.db"

print("Setup completed successfully.")

Setup completed successfully.


In [ ]:
def get_soup(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

print("get_soup function ready.")

get_soup function ready.


In [ ]:
def scrape_all_products():

    books = []

    for page_number in range(1, PAGES_TO_SCRAPE + 1):

        page_url = ALL_PRODUCTS_URL.format(page_number)

        print(f"Scraping catalogue page: {page_number}")

        soup = get_soup(page_url)

        products = soup.select("article.product_pod")

        print(f"Books found: {len(products)}")

        for product in products:

            # Title
            title = product.h3.a.get("title")

            # Price
            price = product.select_one(
                ".price_color"
            ).get_text(strip=True)

            # Rating
            rating_element = product.select_one(
                "p.star-rating"
            )

            star_rating = rating_element.get("class")[1]

            # Availability
            availability = product.select_one(
                ".availability"
            ).get_text(" ", strip=True)

            # Book URL
            book_url = urljoin(
                page_url,
                product.h3.a.get("href")
            )

            # Detail page
            book_soup = get_soup(book_url)

            breadcrumb = book_soup.select(
                "ul.breadcrumb li a"
            )

            category = breadcrumb[-1].get_text(strip=True)

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

    return books

print("Scraping function ready.")

Scraping function ready.


In [ ]:
print("Starting scraping...")

all_books = scrape_all_products()

raw_df = pd.DataFrame(all_books)

print("\nTotal books scraped:", len(raw_df))

display(raw_df.head())

Starting scraping...
Scraping catalogue page: 1
Books found: 20
Scraping catalogue page: 2
Books found: 20
Scraping catalogue page: 3
Books found: 20
Scraping catalogue page: 4
Books found: 20
Scraping catalogue page: 5
Books found: 20

Total books scraped: 100


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


In [ ]:
# --------------------------------------------------
# SELECT 60 BOOKS FROM AT LEAST 3 CATEGORIES
# --------------------------------------------------

# Check available categories
print("Available categories:")
print(raw_df["category"].value_counts())

# Select the first 60 books
df = raw_df.head(REQUIRED_BOOKS).copy()

# Check category count
category_count = df["category"].nunique()

print("\nSelected books:", len(df))
print("Number of categories:", category_count)

# Required checks
assert len(df) >= 60, "Dataset contains fewer than 60 books."
assert category_count >= 3, "Dataset contains fewer than 3 categories."

print("\nDataset satisfies the minimum requirement.")

Available categories:
category
Sequential Art        14
Nonfiction            12
Default                9
Poetry                 7
Fiction                5
Food and Drink         5
Add a comment          5
Young Adult            4
History                4
Fantasy                4
Mystery                3
Music                  3
Childrens              3
Thriller               3
Philosophy             2
Romance                2
Spirituality           2
Science Fiction        2
Politics               1
Business               1
Historical Fiction     1
Travel                 1
Art                    1
Contemporary           1
New Adult              1
Science                1
Health                 1
Horror                 1
Self Help              1
Name: count, dtype: int64

Selected books: 60
Number of categories: 25

Dataset satisfies the minimum requirement.


In [ ]:
# Select 60 books and create exactly 3 categories

# Make a copy
df_60 = df.copy()

# Create 3 broad categories
def create_category(category):
    if category in ['Fiction', 'Fantasy', 'Mystery', 'Romance', 'Young Adult',
                    'Sequential Art', 'Poetry']:
        return 'Fiction'

    elif category in ['Nonfiction', 'History', 'Biography', 'Science',
                      'Business', 'Psychology']:
        return 'Nonfiction'

    else:
        return 'Other'


df_60['category'] = df_60['category'].apply(create_category)

# Keep exactly 60 books
df_60 = df_60.head(60).copy()

# Check result
print("Total books:", len(df_60))
print("Total categories:", df_60['category'].nunique())

print("\nCategories:")
print(df_60['category'].value_counts())

display(df_60.head())

Total books: 60
Total categories: 3

Categories:
category
Other         28
Fiction       23
Nonfiction     9
Name: count, dtype: int64


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Fiction
1,Tipping the Velvet,Â£53.74,One,In stock,Other
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Fiction
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,Nonfiction


In [ ]:
# Make a copy of original dataframe
df_60 = df.copy()

# Create 3 broad categories
def create_category(category):
    if category in ['Fiction', 'Fantasy', 'Mystery', 'Romance', 'Young Adult',
                    'Sequential Art', 'Poetry']:
        return 'Fiction'

    elif category in ['Nonfiction', 'History', 'Biography', 'Science',
                      'Business', 'Psychology']:
        return 'Nonfiction'

    else:
        return 'Other'


# Apply the new categories
df_60['category'] = df_60['category'].apply(create_category)

# Check how many books are available in each category
print(df_60['category'].value_counts())

category
Other         28
Fiction       23
Nonfiction     9
Name: count, dtype: int64


In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# Make a fresh copy of the 60-book dataset
clean_df = df_60.copy()

# Clean price safely
# Extract only the numeric part from values such as £51.77 or Â£51.77
clean_df["price_gbp"] = (
    clean_df["price"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

# Convert star rating text to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

clean_df["rating"] = clean_df["star_rating"].map(rating_map)

# Convert availability to True/False
clean_df["in_stock"] = clean_df["availability"].astype(str).str.contains(
    "In stock",
    case=False,
    na=False
)

# Required fixed conversion rate
GBP_TO_INR = 105.50

# Convert GBP to INR
clean_df["price_inr"] = clean_df["price_gbp"] * GBP_TO_INR

# Keep only required columns
clean_df = clean_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
]

print("Total books:", len(clean_df))
print("Total categories:", clean_df["category"].nunique())

display(clean_df.head())

Total books: 60
Total categories: 3


,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.735,3,True,Fiction
1,Tipping the Velvet,53.74,5669.570,1,True,Other
2,Soumission,50.10,5285.550,1,True,Fiction
3,Sharp Objects,47.82,5045.010,4,True,Fiction
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,Nonfiction


In [ ]:
print(clean_df.dtypes)

title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [ ]:
print(clean_df.isnull().sum())

title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64


In [ ]:
# Database name
DB_NAME = "books.db"

# Connect to SQLite database
conn = sqlite3.connect(DB_NAME)

# Enable foreign key support
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

# Remove old tables if they already exist
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

# Create categories table
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Create books table
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

print("Database and tables created successfully.")

Database and tables created successfully.


In [ ]:
# Get unique categories
categories = clean_df["category"].unique()

for category in categories:
    cursor.execute(
        "INSERT INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

# Check categories
category_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

display(category_df)

,category_id,category_name
0,1,Fiction
1,2,Other
2,3,Nonfiction


In [ ]:
# Create category -> category_id mapping
category_mapping = dict(
    zip(
        category_df["category_name"],
        category_df["category_id"]
    )
)

# Insert books
for _, row in clean_df.iterrows():

    category_id = category_mapping[row["category"]]

    cursor.execute("""
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

print("Books inserted successfully.")

Books inserted successfully.


In [ ]:
import pandas as pd

In [ ]:
result = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

display(result)

,total_books
0,60


In [ ]:
result = pd.read_sql(
    """
    SELECT c.category_name, COUNT(b.book_id) AS book_count
    FROM categories c
    JOIN books b
    ON c.category_id = b.category_id
    GROUP BY c.category_name
    """,
    conn
)

display(result)

,category_name,book_count
0,Fiction,23
1,Nonfiction,9
2,Other,28


In [ ]:
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
"""

result1 = pd.read_sql(query1, conn)

print("QUERY 1:")
print(query1)

display(result1)

QUERY 1:

SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4



,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4
5,Set Me Free,17.46,5
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Rip it Up and Start Again,35.02,5
8,Chase Me (Paris Nights #2),25.27,5
9,Black Dust,34.53,5


In [ ]:
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

print("QUERY 2:")
print(query2)

display(result2)

QUERY 2:

SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10



,title,price_gbp,rating
0,Slow States of Collapse: Poems,57.31,3
1,Our Band Could Be Your Life: Scenes from the A...,57.25,3
2,The Past Never Ends,56.50,4
3,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
4,The Secret of Dreadwillow Carse,56.13,1
5,Birdsong: A Story in Pictures,54.64,3
6,Sapiens: A Brief History of Humankind,54.23,5
7,Tipping the Velvet,53.74,1
8,Aladdin and His Wonderful Lamp,53.13,3
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5


In [ ]:
query3 = """
SELECT DISTINCT category_name
FROM categories
"""

result3 = pd.read_sql(query3, conn)

print("QUERY 3:")
print(query3)

display(result3)

QUERY 3:

SELECT DISTINCT category_name
FROM categories



,category_name
0,Fiction
1,Other
2,Nonfiction


In [ ]:
query4 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
"""

result4 = pd.read_sql(query4, conn)

print("QUERY 4:")
print(query4)

display(result4)

QUERY 4:

SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40



,title,price_gbp,price_inr
0,The Requiem Red,22.65,2389.575
1,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.370
2,The Boys in the Boat: Nine Americans and Their...,22.60,2384.300
3,Shakespeare's Sonnets,20.66,2179.630
4,Rip it Up and Start Again,35.02,3694.610
5,Olio,23.88,2519.340
6,Mesaerion: The Best Science Fiction Stories 18...,37.59,3965.745
7,How Music Works,37.32,3937.260
8,Foolproof Preserving: A Guide to Small Batch J...,30.52,3219.860
9,Chase Me (Paris Nights #2),25.27,2665.985


In [ ]:
# --------------------------------------------------
# QUERY 5 - IN
# --------------------------------------------------

query5 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC
"""

result5 = pd.read_sql(query5, conn)

print("=" * 70)
print("QUERY 5: Books with rating 4 or 5")
print("=" * 70)

print(query5)

display(result5)

QUERY 5: Books with rating 4 or 5

SELECT title, price_gbp, rating
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC



,title,price_gbp,rating
0,Sapiens: A Brief History of Humankind,54.23,5
1,Set Me Free,17.46,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
3,Rip it Up and Start Again,35.02,5
4,Chase Me (Paris Nights #2),25.27,5
5,Black Dust,34.53,5
6,Worlds Elsewhere: Journeys Around Shakespeareâ...,40.30,5
7,The Four Agreements: A Practical Guide to Pers...,17.66,5
8,The Elephant Tree,23.82,5
9,Sophie's World,15.94,5


In [ ]:
# --------------------------------------------------
# SQL QUERY 6
# JOIN
# --------------------------------------------------

join_query = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating,
    b.in_stock
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title
LIMIT 10
"""

join_result = pd.read_sql(join_query, conn)

print("=" * 70)
print("QUERY 6: JOIN - Top 10 Highest-Rated Books")
print("=" * 70)
print(join_query)
display(join_result)

QUERY 6: JOIN - Top 10 Highest-Rated Books

SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating,
    b.in_stock
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title
LIMIT 10



,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# Read Query 1 result using pd.read_sql()
sql_result_1 = pd.read_sql(query1, conn)

# Read Query 2 result using pd.read_sql()
sql_result_2 = pd.read_sql(query2, conn)

print("Query 1 result using pd.read_sql():")
display(sql_result_1)

print("\nQuery 2 result using pd.read_sql():")
display(sql_result_2)

Query 1 result using pd.read_sql():


,title,price_gbp,rating
0,Sharp Objects,47.82,4
1,Sapiens: A Brief History of Humankind,54.23,5
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4
3,The Boys in the Boat: Nine Americans and Their...,22.60,4
4,Shakespeare's Sonnets,20.66,4
5,Set Me Free,17.46,5
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Rip it Up and Start Again,35.02,5
8,Chase Me (Paris Nights #2),25.27,5
9,Black Dust,34.53,5



Query 2 result using pd.read_sql():


,title,price_gbp,rating
0,Slow States of Collapse: Poems,57.31,3
1,Our Band Could Be Your Life: Scenes from the A...,57.25,3
2,The Past Never Ends,56.50,4
3,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
4,The Secret of Dreadwillow Carse,56.13,1
5,Birdsong: A Story in Pictures,54.64,3
6,Sapiens: A Brief History of Humankind,54.23,5
7,Tipping the Velvet,53.74,1
8,Aladdin and His Wonderful Lamp,53.13,3
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5


In [ ]:
# Read books table
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

# Read categories table
categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print("Books DataFrame:")
display(books_df.head())

print("\nCategories DataFrame:")
display(categories_df)

Books DataFrame:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,A Light in the Attic,51.77,5461.735,3,1,1
1,2,Tipping the Velvet,53.74,5669.570,1,1,2
2,3,Soumission,50.10,5285.550,1,1,1
3,4,Sharp Objects,47.82,5045.010,4,1,1
4,5,Sapiens: A Brief History of Humankind,54.23,5721.265,5,1,3



Categories DataFrame:


,category_id,category_name
0,1,Fiction
1,2,Other
2,3,Nonfiction


In [ ]:
# --------------------------------------------------
# REPRODUCE SQL JOIN USING pd.merge()
# --------------------------------------------------

pandas_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Select the same columns as SQL JOIN
pandas_result = pandas_result[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
]

# Apply the same sorting and LIMIT as SQL
pandas_result = pandas_result.sort_values(
    by=["rating", "title"],
    ascending=[False, True]
).head(10).reset_index(drop=True)

# Reset SQL result index
sql_result = join_result.reset_index(drop=True)

print("SQL JOIN RESULT:")
display(sql_result)

print("\nPANDAS MERGE RESULT:")
display(pandas_result)

SQL JOIN RESULT:


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1



PANDAS MERGE RESULT:


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# Save cleaned dataset
df.to_csv("cleaned_books.csv", index=False)

# Save SQL query outputs
result1.to_csv("query1_output.csv", index=False)
result2.to_csv("query2_output.csv", index=False)
result3.to_csv("query3_output.csv", index=False)
result4.to_csv("query4_output.csv", index=False)
result5.to_csv("query5_output.csv", index=False)
join_result.to_csv("join_query_output.csv", index=False)

# Save the SQL queries themselves
with open("sql_queries.txt", "w") as f:

    f.write("QUERY 1 - SELECT + WHERE\n")
    f.write(query1)

    f.write("\n\nQUERY 2 - ORDER BY + LIMIT\n")
    f.write(query2)

    f.write("\n\nQUERY 3 - DISTINCT\n")
    f.write(query3)

    f.write("\n\nQUERY 4 - BETWEEN\n")
    f.write(query4)

    f.write("\n\nQUERY 5 - IN\n")
    f.write(query5)

    f.write("\n\nQUERY 6 - JOIN\n")
    f.write(join_query)

print("All outputs saved successfully.")

All outputs saved successfully.


In [ ]:

# ==========================================
# QUERY 6 - SQL JOIN
# ==========================================

query6 = """
SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10
"""

result6 = pd.read_sql(query6, conn)

print("QUERY 6 - JOIN")
print(query6)

display(result6)

QUERY 6 - JOIN

SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC
LIMIT 10



,title,price_gbp,price_inr,rating,in_stock,category_name
0,Sapiens: A Brief History of Humankind,54.23,5721.265,5,1,Nonfiction
1,Set Me Free,17.46,1842.030,5,1,Fiction
2,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.595,5,1,Fiction
3,Rip it Up and Start Again,35.02,3694.610,5,1,Other
4,Chase Me (Paris Nights #2),25.27,2665.985,5,1,Fiction
5,Black Dust,34.53,3642.915,5,1,Fiction
6,Worlds Elsewhere: Journeys Around Shakespeareâ...,40.30,4251.650,5,1,Nonfiction
7,The Four Agreements: A Practical Guide to Pers...,17.66,1863.130,5,1,Other
8,The Elephant Tree,23.82,2513.010,5,1,Other
9,Sophie's World,15.94,1681.670,5,1,Other


In [ ]:
# ============================================================
# PANDAS MERGE - REPRODUCE THE SQL JOIN
# ============================================================

# Read the two database tables into pandas
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print("BOOKS TABLE")
display(books_df.head())

print("CATEGORIES TABLE")
display(categories_df)


# ------------------------------------------------------------
# Perform the JOIN using pandas.merge()
# ------------------------------------------------------------

merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)


# ------------------------------------------------------------
# Select exactly the same columns used in SQL JOIN
# ------------------------------------------------------------

merge_result = merge_result[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
]


# ------------------------------------------------------------
# Make sure in_stock has the same type as SQL output
# ------------------------------------------------------------

merge_result["in_stock"] = merge_result["in_stock"].astype(int)


# ------------------------------------------------------------
# Apply the SAME sorting as SQL
# ------------------------------------------------------------

merge_result = merge_result.sort_values(
    by=["rating", "title"],
    ascending=[False, True]
)


# ------------------------------------------------------------
# Apply the SAME LIMIT 10 as SQL
# ------------------------------------------------------------

merge_result = merge_result.head(10)


# ------------------------------------------------------------
# Reset index so comparison is fair
# ------------------------------------------------------------

merge_result = merge_result.reset_index(drop=True)


# ------------------------------------------------------------
# Prepare SQL result
# ------------------------------------------------------------

sql_result = join_result[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
].copy()

sql_result["in_stock"] = sql_result["in_stock"].astype(int)

sql_result = sql_result.reset_index(drop=True)


# ------------------------------------------------------------
# DISPLAY BOTH RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("SQL JOIN RESULT")
print("=" * 70)

display(sql_result)


print("=" * 70)
print("PANDAS MERGE RESULT")
print("=" * 70)

display(merge_result)

BOOKS TABLE


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,A Light in the Attic,51.77,5461.735,3,1,1
1,2,Tipping the Velvet,53.74,5669.570,1,1,2
2,3,Soumission,50.10,5285.550,1,1,1
3,4,Sharp Objects,47.82,5045.010,4,1,1
4,5,Sapiens: A Brief History of Humankind,54.23,5721.265,5,1,3


CATEGORIES TABLE


,category_id,category_name
0,1,Fiction
1,2,Other
2,3,Nonfiction


SQL JOIN RESULT


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


PANDAS MERGE RESULT


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# ============================================================
# STEP 1: SQL JOIN RESULT
# ============================================================

join_query = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title ASC
LIMIT 10
"""

sql_result = pd.read_sql(join_query, conn)

print("SQL JOIN RESULT")
display(sql_result)

SQL JOIN RESULT


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# ============================================================
# STEP 2: PANDAS MERGE RESULT
# ============================================================

# Read both tables
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

# Perform INNER JOIN using pandas
merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Select exactly the same columns as SQL
merge_result = merge_result[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
]

# Same sorting as SQL
merge_result = merge_result.sort_values(
    by=["rating", "title"],
    ascending=[False, True]
)

# Same LIMIT 10 as SQL
merge_result = merge_result.head(10)

# Reset index
merge_result = merge_result.reset_index(drop=True)

print("PANDAS MERGE RESULT")
display(merge_result)

PANDAS MERGE RESULT


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# ============================================================
# STEP 3: NORMALIZE BOTH RESULTS
# ============================================================

sql_compare = sql_result.copy()
pandas_compare = merge_result.copy()

# ------------------------------------------------------------
# Make title and category text identical
# ------------------------------------------------------------

sql_compare["title"] = sql_compare["title"].astype(str).str.strip()
pandas_compare["title"] = pandas_compare["title"].astype(str).str.strip()

sql_compare["category_name"] = (
    sql_compare["category_name"].astype(str).str.strip()
)

pandas_compare["category_name"] = (
    pandas_compare["category_name"].astype(str).str.strip()
)

# ------------------------------------------------------------
# Convert price to numeric and round
# ------------------------------------------------------------

sql_compare["price_gbp"] = pd.to_numeric(
    sql_compare["price_gbp"],
    errors="coerce"
).round(2)

pandas_compare["price_gbp"] = pd.to_numeric(
    pandas_compare["price_gbp"],
    errors="coerce"
).round(2)

# ------------------------------------------------------------
# Convert rating to integer
# ------------------------------------------------------------

sql_compare["rating"] = pd.to_numeric(
    sql_compare["rating"],
    errors="coerce"
).astype("Int64")

pandas_compare["rating"] = pd.to_numeric(
    pandas_compare["rating"],
    errors="coerce"
).astype("Int64")

# ------------------------------------------------------------
# Convert in_stock to integer
# SQLite stores BOOLEAN as INTEGER (0/1)
# ------------------------------------------------------------

sql_compare["in_stock"] = (
    pd.to_numeric(
        sql_compare["in_stock"],
        errors="coerce"
    ).astype("Int64")
)

pandas_compare["in_stock"] = (
    pandas_compare["in_stock"]
    .astype(bool)
    .astype(int)
    .astype("Int64")
)

# ------------------------------------------------------------
# Reset index
# ------------------------------------------------------------

sql_compare = sql_compare.reset_index(drop=True)
pandas_compare = pandas_compare.reset_index(drop=True)

print("SQL comparison DataFrame:")
display(sql_compare)

print("Pandas comparison DataFrame:")
display(pandas_compare)

SQL comparison DataFrame:


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


Pandas comparison DataFrame:


,title,category_name,price_gbp,rating,in_stock
0,#HigherSelfie: Wake Up Your Life. Free Your So...,Nonfiction,23.11,5,1
1,Black Dust,Fiction,34.53,5,1
2,Chase Me (Paris Nights #2),Fiction,25.27,5,1
3,Private Paris (Private #10),Fiction,47.61,5,1
4,Rip it Up and Start Again,Other,35.02,5,1
5,Sapiens: A Brief History of Humankind,Nonfiction,54.23,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,Fiction,52.29,5,1
7,Set Me Free,Fiction,17.46,5,1
8,Sophie's World,Other,15.94,5,1
9,The Elephant Tree,Other,23.82,5,1


In [ ]:
# ============================================================
# STEP 4: FINAL COMPARISON
# ============================================================

comparison_result = sql_compare.equals(pandas_compare)

print("=" * 70)
print("FINAL JOIN VALIDATION")
print("=" * 70)

print("SQL JOIN and pandas merge equivalent:", comparison_result)

FINAL JOIN VALIDATION
SQL JOIN and pandas merge equivalent: True


# Data Pipeline

## Overview

This module implements an end-to-end data engineering pipeline for
scraping, cleaning, converting, storing, and querying book catalogue data.

The data is collected from Books to Scrape, a public website designed
for scraping practice.

## Data Collection

The pipeline scrapes the first five pages of the All Products catalogue.

Each page contains approximately 20 books, resulting in 100 scraped
books. From this dataset, 60 books are selected for the project.

The selected dataset contains at least three different book categories.

## Libraries Used

- requests
- BeautifulSoup
- pandas
- sqlite3

## Data Cleaning

The following transformations are applied:

1. The pound symbol is removed from the price field.
2. The cleaned price is converted to a float column named price_gbp.
3. Star ratings such as One, Two, Three, Four, and Five are converted
   into integers from 1 to 5.
4. Availability text is converted into a boolean column named in_stock.
5. If a numeric field fails to parse, the median value is used for
   imputation.

## Currency Conversion

The project uses the required fixed conversion rate:

1 GBP = 105.50 INR

The INR price is calculated as:

price_inr = price_gbp * 105.50

This is a project-defined constant and does not require an external
currency API.

## Database Design

The SQLite database contains two normalized tables.

### categories

- category_id: Primary Key
- category_name: Unique category name

### books

- book_id: Primary Key
- title
- price_gbp
- price_inr
- rating
- in_stock
- category_id: Foreign Key referencing categories

The category information is stored separately to avoid unnecessary
repetition and to maintain a normalized relational structure.

## SQL Queries

The pipeline executes six SQL queries covering:

- SELECT and WHERE
- ORDER BY
- LIMIT
- DISTINCT
- BETWEEN
- IN
- JOIN

The SQL query strings and their outputs are saved in the repository.

## Pandas Validation

At least two SQL query results are read using pd.read_sql().

The JOIN query is reproduced using pd.merge() on the in-memory
books and categories DataFrames.

The SQL JOIN result and pandas merge result are compared using:

sql_result.equals(pandas_result)

The results are equivalent.

## Output Files

- books.db
- cleaned_books.csv
- query1_output.csv
- query2_output.csv
- query3_output.csv
- query4_output.csv
- query5_output.csv
- join_query_output.csv
- sql_queries.txt

## How to Run

Install the required libraries:

pip install -r requirements.txt

Run the pipeline:

python data_pipeline.py

your-project/
│
├── README.md
│
├── data_pipeline/
│   ├── data_pipeline.py
│   ├── requirements.txt
│   ├── README.md
│   ├── books.db
│   ├── cleaned_books.csv
│   ├── query1_output.csv
│   ├── query2_output.csv
│   ├── query3_output.csv
│   ├── query4_output.csv
│   ├── query5_output.csv
│   ├── join_query_output.csv
│   └── sql_queries.txt
│
├── analytics/
│
└── support_assistant/